# Master Script — CLUSTERIZAÇÃO (scikit-learn)

**Guia de consulta.** Não foi feito para rodar de ponta a ponta: é um repositório de
sintaxe, pipelines e grades de hiperparâmetros para copiar quando um projeto novo começa.

### Fluxo do documento
| # | Etapa | O que tem aqui |
|---|-------|----------------|
| 0 | Setup | Imports canônicos, `RANDOM_STATE`, config global |
| 1 | Data Prep | Leitura, tipagem, outliers, escala das variáveis |
| 2 | Split | Holdout para **estabilidade** (o papel do split muda aqui) |
| 3 | Pipelines | `ColumnTransformer` + um pipeline por família + redução de dimensão |
| 4 | Validação | Métricas internas, curva do cotovelo, **estabilidade por reamostragem** |
| 5 | Otimização | `RandomizedSearchCV` com scorer próprio + `ParameterSampler` |
| 6 | Prova Final | Perfil dos clusters, projeção 2D, tabela de contingência, ARI/NMI |
| A | Apêndice | Cheatsheet de famílias, métricas e armadilhas |

### As quatro diferenças em relação ao aprendizado supervisionado
1. **Não existe `y`.** Todo `fit` é `fit(X)`; não há gabarito para acertar.
2. **O "split" muda de função.** Não serve para medir acerto — serve para verificar se a estrutura encontrada **se repete** em dados que o modelo não viu. Vários algoritmos (DBSCAN, Agglomerative, OPTICS) nem têm `predict`.
3. **A "validação cruzada" é estabilidade.** Reamostra, roda de novo, mede o quanto as partições concordam (ARI). Cluster que muda a cada rodada não é cluster, é ruído.
4. **A escala domina o resultado.** K-Means e DBSCAN são baseados em distância: sem `StandardScaler`, a variável de maior amplitude define os grupos sozinha.

---
## 0. Setup e imports

In [ ]:
# --- Núcleo ---------------------------------------------------------------
import numpy as np
import pandas as pd

# --- Pré-processamento ----------------------------------------------------
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import (
    StandardScaler,      # padrão para K-Means / DBSCAN / hierárquico
    RobustScaler,        # quando há outliers
    MinMaxScaler,        # quando o algoritmo assume domínio limitado
    PowerTransformer, QuantileTransformer, Normalizer,   # Normalizer = norma L2 por LINHA
    OneHotEncoder, OrdinalEncoder,
)
from sklearn.decomposition import PCA, TruncatedSVD, NMF, FastICA
from sklearn.manifold import TSNE, Isomap, SpectralEmbedding

# --- Algoritmos de clusterização: famílias matemáticas --------------------
from sklearn.cluster import (
    KMeans, MiniBatchKMeans, BisectingKMeans,      # centróide (partição)
    AgglomerativeClustering,                        # hierárquico
    DBSCAN, HDBSCAN, OPTICS,                        # densidade
    MeanShift, estimate_bandwidth,                  # deslocamento de média
    SpectralClustering,                             # grafo / espectral
    Birch,                                          # incremental (grandes volumes)
    AffinityPropagation,                            # troca de mensagens
)
from sklearn.mixture import GaussianMixture, BayesianGaussianMixture   # probabilístico

# --- Métricas -------------------------------------------------------------
from sklearn.metrics import (
    # INTERNAS (só usam X e os rótulos previstos) — o caso normal
    silhouette_score, silhouette_samples,
    calinski_harabasz_score, davies_bouldin_score,
    # EXTERNAS (exigem um rótulo verdadeiro) — só quando existe gabarito
    adjusted_rand_score, normalized_mutual_info_score, adjusted_mutual_info_score,
    homogeneity_completeness_v_measure, fowlkes_mallows_score,
    confusion_matrix,
)
from sklearn.metrics.cluster import contingency_matrix, pair_confusion_matrix
from sklearn.neighbors import NearestNeighbors

from sklearn.model_selection import (
    train_test_split, RandomizedSearchCV, GridSearchCV, ParameterSampler, ParameterGrid
)
from sklearn.base import clone

import matplotlib.pyplot as plt
import joblib

# --- Config global --------------------------------------------------------
RANDOM_STATE = 42
N_JOBS = -1
pd.set_option("display.max_columns", 100)
np.random.seed(RANDOM_STATE)

# --- Opcionais úteis ------------------------------------------------------
# pip install umap-learn scipy kneed
# import umap                                   # projeção não-linear, melhor que t-SNE p/ estrutura global
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist

---
## 1. Carregamento e Limpeza (Data Prep)

Esqueleto do preparo padrão. Dados fictícios com **três grupos plantados**, para que a
etapa 6 tenha um gabarito de referência — situação que na prática você não terá.

In [ ]:
# ==== 1.1 Leitura ==========================================================
# df = pd.read_csv("dados.csv", sep=",", decimal=".", encoding="utf-8")
# df = pd.read_parquet("dados.parquet")

# ---- DataFrame fictício (substituir pela leitura real) --------------------
n = 900
rng = np.random.default_rng(RANDOM_STATE)

# Três perfis de cliente com centros distintos
centros = {
    0: dict(gasto=(300, 80),   freq=(2, 0.8),  ticket=(150, 40),  recencia=(60, 20)),
    1: dict(gasto=(1500, 300), freq=(12, 3),   ticket=(130, 35),  recencia=(10, 5)),
    2: dict(gasto=(800, 200),  freq=(5, 1.5),  ticket=(400, 90),  recencia=(30, 12)),
}
blocos, grupo_real = [], []
for g, (k, p) in enumerate(centros.items()):
    m = n // 3
    blocos.append(pd.DataFrame({
        var: rng.normal(mu, sd, m) for var, (mu, sd) in p.items()
    }))
    grupo_real += [g] * m

df = pd.concat(blocos, ignore_index=True)
df["gasto_total"]   = df.pop("gasto").clip(1)
df["frequencia"]    = df.pop("freq").clip(1)
df["ticket_medio"]  = df.pop("ticket").clip(1)
df["recencia_dias"] = df.pop("recencia").clip(0)
df["canal"]   = rng.choice(["app", "web", "loja"], len(df))
df["regiao"]  = rng.choice(["Norte", "Sul", "Sudeste"], len(df))
df["id_cliente"] = np.arange(len(df))

# Guardado SÓ para conferência na etapa 6. Nunca entra no fit.
GRUPO_REAL = pd.Series(grupo_real, name="grupo_real")

df = df.sample(frac=1, random_state=RANDOM_STATE)   # embaralha
GRUPO_REAL = GRUPO_REAL.loc[df.index].reset_index(drop=True)
df = df.reset_index(drop=True)

for col, frac in [("ticket_medio", .04), ("canal", .03)]:
    df.loc[df.sample(frac=frac, random_state=RANDOM_STATE).index, col] = np.nan

df.head()

In [ ]:
# ==== 1.2 Diagnóstico ======================================================
print(df.shape)
df.info()
display(df.describe().T)
print(df.isna().mean().mul(100).round(2))

# A escala bruta das variáveis: quem tem amplitude maior domina a distância
display(df.select_dtypes(np.number).agg(["min", "max", "mean", "std"]).T.round(2))

# Distribuições: cauda longa distorce o centróide do K-Means
df.select_dtypes(np.number).hist(bins=40, figsize=(12, 6)); plt.tight_layout(); plt.show()

# Correlação: variáveis redundantes pesam duas vezes na distância
display(df.select_dtypes(np.number).corr().round(3))

In [ ]:
# ==== 1.3 Limpeza padrão ==================================================
# a) Identificadores fora — id numérico vira variável e destrói o agrupamento
df = df.drop(columns=["id_cliente"], errors="ignore")

# b) Duplicatas
df = df.drop_duplicates()

# c) Tipagem
CAT_COLS = ["canal", "regiao"]
NUM_COLS = ["gasto_total", "frequencia", "ticket_medio", "recencia_dias"]
df[CAT_COLS] = df[CAT_COLS].astype("category")
df[NUM_COLS] = df[NUM_COLS].apply(pd.to_numeric, errors="coerce")

# d) Outliers: em clusterização eles são decisivos.
#      K-Means      -> puxa o centróide, pode gastar um cluster inteiro num ponto
#      DBSCAN/HDBSCAN -> classifica como ruído (rótulo -1) sozinho, sem intervenção
#    Opções: RobustScaler / QuantileTransformer, ou usar algoritmo de densidade.
z = (df[NUM_COLS] - df[NUM_COLS].mean()) / df[NUM_COLS].std()
print(f"linhas com |z| > 4 em alguma variável: {(z.abs() > 4).any(axis=1).sum()}")
# df = df[~(z.abs() > 4).any(axis=1)]     # descomentar só se for erro de medição

# e) X — não há y.
X = df.copy()
print(X.shape)

---
## 2. Separação dos Dados (Split)

Aqui o split muda de função. Não existe acerto a medir; o que se testa é se a estrutura
encontrada **se sustenta fora da amostra usada para achá-la**.

| Uso do holdout | Como | Vale para |
|---|---|---|
| **Generalização** — ajusta no treino, atribui no holdout e compara a silhueta | `fit(X_train)` → `predict(X_test)` | KMeans, MiniBatch, Bisecting, GMM, Birch, MeanShift |
| **Estabilidade** — reamostra, roda de novo, mede concordância (ARI) | duas subamostras com sobreposição | **todos**, inclusive DBSCAN e Agglomerative |
| **Sem holdout** — agrupa a base inteira de uma vez | `fit_predict(X)` | quando o objetivo é rotular *esta* base, não prever novas linhas |

> `AgglomerativeClustering`, `DBSCAN`, `OPTICS`, `SpectralClustering` e
> `AffinityPropagation` **não têm `predict`** — só `fit_predict`. Para eles, a validação
> é por estabilidade (2.2) ou por um classificador treinado sobre os rótulos (6.6).

In [ ]:
# ==== 2.1 Holdout de generalização (para algoritmos com predict) ==========
X_train, X_test = train_test_split(
    X, test_size=0.20, random_state=RANDOM_STATE, shuffle=True
)
print(f"treino: {X_train.shape} | holdout: {X_test.shape}")

# Estratificar por uma variável de negócio conhecida mantém a composição:
# X_train, X_test = train_test_split(X, test_size=.2, stratify=X["regiao"],
#                                    random_state=RANDOM_STATE)

In [ ]:
# ==== 2.2 Amostras para o teste de estabilidade ===========================
# Duas subamostras de 80% com sobreposição. Rodar o mesmo algoritmo nas duas e
# comparar os rótulos na interseção mede se a estrutura é real ou fruto do acaso.
def duas_subamostras(X, frac=0.8, seed=0):
    rs = np.random.RandomState(seed)
    idx_a = rs.choice(X.index, int(len(X) * frac), replace=False)
    idx_b = rs.choice(X.index, int(len(X) * frac), replace=False)
    comum = np.intersect1d(idx_a, idx_b)
    return X.loc[idx_a], X.loc[idx_b], comum

Xa, Xb, comum = duas_subamostras(X_train, seed=RANDOM_STATE)
print(f"subamostra A: {len(Xa)} | B: {len(Xb)} | interseção: {len(comum)}")

In [ ]:
# ==== 2.3 Amostragem para algoritmos caros ================================
# Complexidade típica (n = nº de linhas):
#   KMeans, MiniBatchKMeans, Birch ............ O(n)      -> milhões de linhas
#   DBSCAN, HDBSCAN, OPTICS ................... O(n log n) -> centenas de milhares
#   AgglomerativeClustering, Spectral ......... O(n²)-O(n³) -> até ~10-20 mil
#   AffinityPropagation ....................... O(n²) em memória -> até ~5 mil
LIMITE_ALGORITMO_CARO = 10_000
X_amostra = (X_train.sample(LIMITE_ALGORITMO_CARO, random_state=RANDOM_STATE)
             if len(X_train) > LIMITE_ALGORITMO_CARO else X_train)
print(f"amostra para algoritmos O(n²): {X_amostra.shape}")

---
## 3. Construção dos Pipelines

Duas receitas de `ColumnTransformer`. Em clusterização praticamente tudo é sensível a
escala — só o `HDBSCAN` sobre dados já homogêneos escapa.

| Receita | Numéricas | Categóricas | Observação |
|---|---|---|---|
| `"scaled"` | mediana + `StandardScaler` | moda + `OneHotEncoder` | padrão para tudo |
| `"robust"` | mediana + `QuantileTransformer` | moda + `OneHotEncoder` | quando há cauda longa/outliers |

### O problema das variáveis categóricas
`OneHotEncoder` cria colunas binárias cuja distância euclidiana é sempre √2 entre
categorias diferentes — o que na prática dilui as numéricas. Três saídas:
1. **Reduzir depois do encoding** (`PCA`/`TruncatedSVD`) — simples e funciona;
2. **Ponderar** os blocos numérico e categórico no `ColumnTransformer`;
3. **Trocar de algoritmo**: `KModes`/`KPrototypes` (`pip install kmodes`) ou distância de Gower.

In [ ]:
# ==== 3.1 Seleção de colunas ==============================================
NUM_FEATURES = ["gasto_total", "frequencia", "ticket_medio", "recencia_dias"]
CAT_FEATURES = ["canal", "regiao"]

In [ ]:
# ==== 3.2 Fábrica de pré-processadores ====================================
def build_preprocessor(kind="scaled", scaler=None,
                       num=NUM_FEATURES, cat=CAT_FEATURES, peso_cat=1.0):
    if kind == "scaled":
        num_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", scaler or StandardScaler()),
        ])
    elif kind == "robust":
        num_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", QuantileTransformer(output_distribution="normal",
                                           random_state=RANDOM_STATE)),
        ])
    else:
        raise ValueError(kind)

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                  min_frequency=0.02, sparse_output=False)),
        ("scaler", StandardScaler()),   # coloca o bloco categórico na mesma ordem de grandeza
    ])

    return ColumnTransformer(
        [("num", num_pipe, num), ("cat", cat_pipe, cat)],
        transformer_weights={"num": 1.0, "cat": peso_cat},   # peso_cat < 1 reduz o efeito do OHE
        remainder="drop",
        verbose_feature_names_out=False,
    ).set_output(transform="pandas")

# Só numéricas (mais limpo quando as categóricas são pouco informativas):
prep_num = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

In [ ]:
# ==== 3.3 Um Pipeline por família matemática ==============================
# O passo final chama-se sempre "model" -> as grades usam o prefixo "model__".
K = 3   # chute inicial; a etapa 4 define o valor

PIPELINES = {

    # ======================= CENTRÓIDE (partição) =========================
    # Minimiza a soma das distâncias ao centro. Assume grupos esféricos, de
    # tamanho parecido, e EXIGE que o k seja informado.
    "KMeans": Pipeline([
        ("prep", build_preprocessor("scaled")),
        ("model", KMeans(n_clusters=K, init="k-means++", n_init=10,
                         random_state=RANDOM_STATE)),
    ]),
    "MiniBatchKMeans": Pipeline([            # milhões de linhas, precisão um pouco menor
        ("prep", build_preprocessor("scaled")),
        ("model", MiniBatchKMeans(n_clusters=K, batch_size=1024, n_init=10,
                                  random_state=RANDOM_STATE)),
    ]),
    "BisectingKMeans": Pipeline([            # divide recursivamente: gera hierarquia
        ("prep", build_preprocessor("scaled")),
        ("model", BisectingKMeans(n_clusters=K, random_state=RANDOM_STATE)),
    ]),

    # ======================= HIERÁRQUICO ==================================
    # Constrói uma árvore de fusões. Dá dendrograma e aceita formatos não esféricos
    # (linkage != ward). Sem predict; O(n²).
    "Agglomerative_ward": Pipeline([
        ("prep", build_preprocessor("scaled")),
        ("model", AgglomerativeClustering(n_clusters=K, linkage="ward")),
    ]),
    "Agglomerative_average": Pipeline([
        ("prep", build_preprocessor("scaled")),
        ("model", AgglomerativeClustering(n_clusters=K, linkage="average",
                                          metric="euclidean")),
    ]),

    # ======================= DENSIDADE ====================================
    # Grupo = região densa. NÃO precisa de k, acha formato arbitrário e marca
    # outliers como ruído (rótulo -1). Sofre com densidades muito diferentes.
    "DBSCAN": Pipeline([
        ("prep", build_preprocessor("scaled")),
        ("model", DBSCAN(eps=0.5, min_samples=5, n_jobs=N_JOBS)),
    ]),
    "HDBSCAN": Pipeline([                    # versão hierárquica: dispensa o eps
        ("prep", build_preprocessor("scaled")),
        ("model", HDBSCAN(min_cluster_size=25, min_samples=None, n_jobs=N_JOBS)),
    ]),
    "OPTICS": Pipeline([                     # lida com densidades variadas
        ("prep", build_preprocessor("scaled")),
        ("model", OPTICS(min_samples=10, xi=0.05, n_jobs=N_JOBS)),
    ]),

    # ======================= PROBABILÍSTICO ===============================
    # Mistura de gaussianas: atribuição SUAVE (probabilidade por cluster) e
    # clusters elípticos de tamanhos diferentes. Dá AIC/BIC para escolher k.
    "GaussianMixture": Pipeline([
        ("prep", build_preprocessor("scaled")),
        ("model", GaussianMixture(n_components=K, covariance_type="full",
                                  n_init=5, random_state=RANDOM_STATE)),
    ]),
    "BayesianGMM": Pipeline([                # poda componentes sozinho: k é um teto
        ("prep", build_preprocessor("scaled")),
        ("model", BayesianGaussianMixture(
            n_components=10, covariance_type="full",
            weight_concentration_prior_type="dirichlet_process",
            n_init=3, random_state=RANDOM_STATE)),
    ]),

    # ======================= GRAFO / ESPECTRAL ============================
    # Agrupa no espaço dos autovetores do grafo de similaridade. Excelente para
    # formatos entrelaçados (anéis, luas). O(n³): só em base pequena.
    "Spectral": Pipeline([
        ("prep", build_preprocessor("scaled")),
        ("model", SpectralClustering(n_clusters=K, affinity="nearest_neighbors",
                                     n_neighbors=10, assign_labels="kmeans",
                                     random_state=RANDOM_STATE, n_jobs=N_JOBS)),
    ]),

    # ======================= DESLOCAMENTO DE MÉDIA ========================
    # Busca as modas da densidade. Descobre o k sozinho; sensível ao bandwidth.
    "MeanShift": Pipeline([
        ("prep", build_preprocessor("scaled")),
        ("model", MeanShift(bandwidth=None, bin_seeding=True, n_jobs=N_JOBS)),
    ]),

    # ======================= INCREMENTAL ==================================
    # Constrói uma árvore de resumos (CF-tree) em uma passada. Ótimo pré-agrupador
    # para bases enormes; costuma ser seguido de KMeans nos subclusters.
    "Birch": Pipeline([
        ("prep", build_preprocessor("scaled")),
        ("model", Birch(n_clusters=K, threshold=0.5, branching_factor=50)),
    ]),

    # ======================= TROCA DE MENSAGENS ===========================
    # Escolhe exemplares reais como centros. Não precisa de k, mas O(n²) e
    # tende a devolver clusters demais.
    "AffinityPropagation": Pipeline([
        ("prep", build_preprocessor("scaled")),
        ("model", AffinityPropagation(damping=0.9, random_state=RANDOM_STATE)),
    ]),
}

print(f"{len(PIPELINES)} pipelines declarados")

In [ ]:
# ==== 3.4 Redução de dimensionalidade dentro do Pipeline ==================
# Por que reduzir: acima de ~10-15 dimensões a distância euclidiana perde poder
# de discriminação (maldição da dimensionalidade) e todo ponto fica "igualmente
# distante" de todos os outros — o cluster deixa de existir matematicamente.

pipe_pca_kmeans = Pipeline([
    ("prep", build_preprocessor("scaled")),
    ("pca", PCA(n_components=0.90, random_state=RANDOM_STATE)),  # 90% da variância
    ("model", KMeans(n_clusters=K, n_init=10, random_state=RANDOM_STATE)),
])

# UMAP + HDBSCAN: combinação padrão atual para estrutura não-linear
# import umap
# pipe_umap_hdbscan = Pipeline([
#     ("prep", build_preprocessor("scaled")),
#     ("umap", umap.UMAP(n_components=5, n_neighbors=15, min_dist=0.0,
#                        random_state=RANDOM_STATE)),
#     ("model", HDBSCAN(min_cluster_size=30)),
# ])

# t-SNE serve para VISUALIZAR (etapa 6), não para agrupar: distorce distâncias
# globais e não tem transform para dados novos.

---
## 4. Avaliação das Famílias

Sem gabarito, a avaliação apoia-se em três pernas — as três precisam concordar antes de
declarar um vencedor.

### (a) Métricas internas
| Métrica | Faixa | Direção | Viés conhecido |
|---|---|---|---|
| **Silhueta** | [-1, 1] | maior | favorece clusters convexos/esféricos |
| **Calinski-Harabasz** | [0, ∞) | maior | favorece k pequeno e grupos compactos |
| **Davies-Bouldin** | [0, ∞) | **menor** | também favorece convexidade |
| **Inércia (SSE)** | [0, ∞) | menor | cai sempre com k → só serve para o cotovelo |
| **BIC / AIC** (só GMM) | ℝ | menor | penaliza complexidade: o critério mais principiado |

### (b) Estabilidade
Reamostra e roda de novo; mede o ARI entre as duas partições na interseção.
**ARI > 0,75 = estrutura sólida. ARI < 0,5 = o algoritmo está inventando os grupos.**

### (c) Utilidade de negócio
Cluster que não se traduz em perfil acionável não vale nada, tenha a silhueta que tiver.
Etapa 6.3.

In [ ]:
# ==== 4.1 Funções de avaliação ============================================
def metricas_internas(X_mat, labels):
    # Ignora o ruído (-1) do DBSCAN/HDBSCAN no cálculo, mas reporta a fração.
    mask = labels != -1
    n_clusters = len(set(labels[mask]))
    frac_ruido = (~mask).mean()
    if n_clusters < 2 or mask.sum() < 3:
        return dict(n_clusters=n_clusters, ruido=frac_ruido,
                    silhueta=np.nan, calinski=np.nan, davies=np.nan)
    Xm = np.asarray(X_mat)[mask]
    lm = labels[mask]
    return dict(
        n_clusters=n_clusters,
        ruido=frac_ruido,
        silhueta=silhouette_score(Xm, lm),
        calinski=calinski_harabasz_score(Xm, lm),
        davies=davies_bouldin_score(Xm, lm),
        menor_cluster=pd.Series(lm).value_counts(normalize=True).min(),
    )


def estabilidade(pipe, X, n_rodadas=10, frac=0.8, seed=RANDOM_STATE):
    # Concordância (ARI) entre pares de execuções em subamostras sobrepostas.
    aris = []
    for i in range(n_rodadas):
        Xa, Xb, comum = duas_subamostras(X, frac=frac, seed=seed + i)
        try:
            la = pd.Series(clone(pipe).fit_predict(Xa), index=Xa.index)
            lb = pd.Series(clone(pipe).fit_predict(Xb), index=Xb.index)
            aris.append(adjusted_rand_score(la.loc[comum], lb.loc[comum]))
        except Exception:
            aris.append(np.nan)
    return np.nanmean(aris), np.nanstd(aris)

In [ ]:
# ==== 4.2 Torneio entre as famílias =======================================
import warnings, time
warnings.filterwarnings("ignore")

# A matriz pré-processada de referência: as métricas internas precisam ser
# calculadas no MESMO espaço para que a comparação faça sentido.
prep_ref = build_preprocessor("scaled").fit(X_train)
X_train_mat = prep_ref.transform(X_train)

linhas = []
for nome, pipe in PIPELINES.items():
    t0 = time.perf_counter()
    try:
        labels = pipe.fit_predict(X_train)
        m = metricas_internas(X_train_mat, np.asarray(labels))
        est_mean, est_std = estabilidade(pipe, X_train, n_rodadas=5)
        m.update(modelo=nome, estabilidade=est_mean, estab_std=est_std,
                 tempo_s=time.perf_counter() - t0)
        linhas.append(m)
        print(f"{nome:<22} k={m['n_clusters']:<3} sil={m['silhueta']:.3f}  "
              f"DB={m['davies']:.3f}  estab={est_mean:.3f}  "
              f"ruido={m['ruido']:.1%}  {m['tempo_s']:.1f}s")
    except Exception as e:
        print(f"{nome:<22} FALHOU: {type(e).__name__}: {e}")

ranking = (pd.DataFrame(linhas)
           .set_index("modelo")
           .sort_values("silhueta", ascending=False))
display(ranking.round(4))

In [ ]:
# ==== 4.3 Curva do cotovelo e escolha do k ================================
# Só para as famílias que exigem k informado (KMeans, GMM, Agglomerative, Birch).
ks = range(2, 12)
inercia, sil, cal, dav, bic = [], [], [], [], []

for k in ks:
    km = Pipeline([("prep", build_preprocessor("scaled")),
                   ("model", KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE))])
    lab = km.fit_predict(X_train)
    inercia.append(km["model"].inertia_)
    sil.append(silhouette_score(X_train_mat, lab))
    cal.append(calinski_harabasz_score(X_train_mat, lab))
    dav.append(davies_bouldin_score(X_train_mat, lab))

    gm = Pipeline([("prep", build_preprocessor("scaled")),
                   ("model", GaussianMixture(n_components=k, covariance_type="full",
                                             n_init=3, random_state=RANDOM_STATE))])
    gm.fit(X_train)
    bic.append(gm["model"].bic(np.asarray(gm["prep"].transform(X_train))))

fig, axes = plt.subplots(1, 5, figsize=(21, 3.6))
for ax, vals, tit in zip(axes,
                         [inercia, sil, cal, dav, bic],
                         ["Inércia (cotovelo)", "Silhueta (máx)",
                          "Calinski-Harabasz (máx)", "Davies-Bouldin (mín)",
                          "BIC do GMM (mín)"]):
    ax.plot(list(ks), vals, "o-")
    ax.set_xlabel("k"); ax.set_title(tit); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

resumo_k = pd.DataFrame({"k": list(ks), "inercia": inercia, "silhueta": sil,
                         "calinski": cal, "davies": dav, "bic_gmm": bic})
display(resumo_k.round(3))

# Detecção automática do cotovelo (pip install kneed):
# from kneed import KneeLocator
# print("cotovelo:", KneeLocator(list(ks), inercia, curve="convex", direction="decreasing").elbow)

K_ESCOLHIDO = int(resumo_k.loc[resumo_k["silhueta"].idxmax(), "k"])
print(f"\nk pela silhueta: {K_ESCOLHIDO}  |  "
      f"k pelo BIC: {int(resumo_k.loc[resumo_k['bic_gmm'].idxmin(), 'k'])}  |  "
      f"k pelo Davies-Bouldin: {int(resumo_k.loc[resumo_k['davies'].idxmin(), 'k'])}")
print("Se os critérios discordarem, prefira o k que gera perfis interpretáveis (etapa 6.3).")

In [ ]:
# ==== 4.4 Ferramentas específicas por família =============================

# --- DBSCAN: escolher o eps pelo gráfico da k-ésima distância -------------
# O joelho da curva é o eps. min_samples: comece com 2 * n_dimensoes.
k_dist = 2 * X_train_mat.shape[1]
nn = NearestNeighbors(n_neighbors=k_dist).fit(X_train_mat)
dists, _ = nn.kneighbors(X_train_mat)
dist_k = np.sort(dists[:, -1])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(dist_k)
ax.set_xlabel("pontos ordenados"); ax.set_ylabel(f"distância ao {k_dist}º vizinho")
ax.set_title("Curva k-distância — o joelho indica o eps"); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()
print(f"sugestão de eps (percentil 95): {np.percentile(dist_k, 95):.3f}")

# --- Hierárquico: dendrograma indica o número de grupos pelo corte --------
Z = linkage(np.asarray(X_train_mat)[:300], method="ward")
fig, ax = plt.subplots(figsize=(12, 4))
dendrogram(Z, truncate_mode="lastp", p=25, ax=ax, no_labels=True)
ax.set_title("Dendrograma (ward) — corte no maior salto vertical")
plt.tight_layout(); plt.show()
# labels = fcluster(Z, t=K_ESCOLHIDO, criterion="maxclust")

# --- MeanShift: estimativa automática do bandwidth ------------------------
bw = estimate_bandwidth(X_train_mat, quantile=0.2, n_samples=500,
                        random_state=RANDOM_STATE)
print(f"bandwidth estimado: {bw:.3f}")

In [ ]:
# ==== 4.5 Escolha da família vencedora ====================================
# Critério composto: exige silhueta razoável E estabilidade alta E clusters
# com tamanho utilizável. Uma silhueta linda com ARI de 0,3 é ruído.
candidatos = ranking[
    (ranking["silhueta"] > 0.20) &
    (ranking["estabilidade"] > 0.60) &
    (ranking["ruido"] < 0.30) &
    (ranking["n_clusters"].between(2, 12))
].copy()

candidatos["score_composto"] = (
    candidatos["silhueta"].rank(pct=True) * 0.4
    + candidatos["estabilidade"].rank(pct=True) * 0.4
    + (-candidatos["davies"]).rank(pct=True) * 0.2
)
display(candidatos.sort_values("score_composto", ascending=False).round(4))

MELHOR_FAMILIA = (candidatos["score_composto"].idxmax()
                  if len(candidatos) else ranking.index[0])
print(f"\nFamília vencedora: {MELHOR_FAMILIA}")

---
## 5. Escolha e Otimização

`RandomizedSearchCV` foi desenhado para aprendizado supervisionado, mas funciona em
clusterização com **um scorer próprio** — a assinatura `scorer(estimator, X, y=None)`
não exige `y`.

Duas rotas, escolha conforme o algoritmo:

| Rota | Quando | Como |
|---|---|---|
| **A — `RandomizedSearchCV` com CV real** | algoritmo tem `predict` (KMeans, GMM, Birch, MiniBatch) | ajusta no fold de treino, pontua no fold de teste → mede generalização de verdade |
| **B — `RandomizedSearchCV` sem CV** | qualquer algoritmo | `cv=[(np.arange(n), np.arange(n))]` ajusta e pontua na base inteira; é só uma varredura organizada |
| **C — `ParameterSampler` manual** | DBSCAN, HDBSCAN, Agglomerative, OPTICS | laço explícito com múltiplas métricas + estabilidade; **o mais informativo** |

> Cuidado com a rota B: sem holdout, maximizar a silhueta tende a escolher o k que a
> silhueta gosta (poucos clusters esféricos), não o k certo. Sempre cruze com estabilidade.

In [ ]:
# ==== 5.1 Scorers para clusterização ======================================
def scorer_silhueta(estimator, X, y=None):
    # Usa predict quando existe (avalia em dados NÃO vistos); senão, fit_predict.
    try:
        labels = estimator.predict(X)
    except (AttributeError, NotImplementedError):
        labels = estimator.fit_predict(X)
    labels = np.asarray(labels)
    mask = labels != -1
    if len(set(labels[mask])) < 2:
        return -1.0                     # partição degenerada: pior nota possível
    Xt = estimator[:-1].transform(X) if isinstance(estimator, Pipeline) else X
    return silhouette_score(np.asarray(Xt)[mask], labels[mask])


def scorer_davies_neg(estimator, X, y=None):
    # Davies-Bouldin é "menor é melhor" -> negativo, porque scorer é sempre "maior é melhor".
    try:
        labels = estimator.predict(X)
    except (AttributeError, NotImplementedError):
        labels = estimator.fit_predict(X)
    labels = np.asarray(labels)
    mask = labels != -1
    if len(set(labels[mask])) < 2:
        return -1e6
    Xt = estimator[:-1].transform(X) if isinstance(estimator, Pipeline) else X
    return -davies_bouldin_score(np.asarray(Xt)[mask], labels[mask])


def scorer_bic_neg(estimator, X, y=None):     # só para GaussianMixture
    Xt = estimator[:-1].transform(X) if isinstance(estimator, Pipeline) else X
    return -estimator["model"].bic(np.asarray(Xt))

SCORING = {"silhueta": scorer_silhueta, "davies_neg": scorer_davies_neg}
METRICA_ALVO = "silhueta"

In [ ]:
# ==== 5.2 Grades de hiperparâmetros — repositório completo ================
from scipy.stats import loguniform, uniform, randint

GRIDS = {

# ============================ CENTRÓIDE ===================================
"KMeans": {
    "model__n_clusters": randint(2, 12),
    "model__init":       ["k-means++", "random"],
    "model__n_init":     [10, 20, 30],        # execuções independentes; fica com a melhor
    "model__max_iter":   [300, 500],
    "model__algorithm":  ["lloyd", "elkan"],
    "model__tol":        loguniform(1e-6, 1e-3),
    # pré-processamento também vale otimizar:
    "prep__num__scaler": [StandardScaler(), RobustScaler(), MinMaxScaler(),
                          PowerTransformer(method="yeo-johnson")],
},

"MiniBatchKMeans": {
    "model__n_clusters":          randint(2, 12),
    "model__batch_size":          [256, 512, 1024, 4096],
    "model__max_no_improvement":  [5, 10, 20],
    "model__reassignment_ratio":  loguniform(1e-3, 1e-1),
    "model__n_init":              [5, 10, 20],
},

"BisectingKMeans": {
    "model__n_clusters":            randint(2, 12),
    "model__bisecting_strategy":    ["biggest_inertia", "largest_cluster"],
    "model__init":                  ["k-means++", "random"],
    "model__n_init":                [1, 5, 10],
},

# ============================ HIERÁRQUICO =================================
"Agglomerative_ward": {
    "model__n_clusters":       randint(2, 12),
    "model__linkage":          ["ward", "complete", "average", "single"],
    # ward só aceita metric="euclidean"; os demais aceitam todas:
    "model__metric":           ["euclidean", "manhattan", "cosine"],
    # alternativa a n_clusters: cortar por altura do dendrograma
    # "model__distance_threshold": uniform(0.5, 5), "model__n_clusters": [None],
},

# ============================ DENSIDADE ===================================
"DBSCAN": {
    "model__eps":         uniform(0.1, 2.0),   # raio da vizinhança — parâmetro crítico
    "model__min_samples": randint(3, 30),      # densidade mínima; comece em 2*n_dim
    "model__metric":      ["euclidean", "manhattan", "cosine"],
    "model__algorithm":   ["auto", "ball_tree", "kd_tree"],
},

"HDBSCAN": {
    "model__min_cluster_size":         randint(5, 100),   # menor grupo aceitável
    "model__min_samples":              [None, 5, 10, 25],  # rigor do que é ruído
    "model__cluster_selection_method": ["eom", "leaf"],    # eom=grupos amplos, leaf=finos
    "model__cluster_selection_epsilon": uniform(0, 1),
    "model__metric":                   ["euclidean", "manhattan"],
},

"OPTICS": {
    "model__min_samples":              randint(5, 50),
    "model__xi":                       uniform(0.01, 0.2),
    "model__min_cluster_size":         uniform(0.01, 0.15),
    "model__cluster_method":           ["xi", "dbscan"],
    "model__max_eps":                  [np.inf, 2.0, 5.0],
},

# ============================ PROBABILÍSTICO ==============================
"GaussianMixture": {
    "model__n_components":    randint(2, 12),
    # full=elipse livre | tied=mesma forma | diag=eixos alinhados | spherical=círculo
    "model__covariance_type": ["full", "tied", "diag", "spherical"],
    "model__init_params":     ["kmeans", "k-means++", "random", "random_from_data"],
    "model__reg_covar":       loguniform(1e-8, 1e-3),   # estabilidade numérica
    "model__n_init":          [3, 5, 10],
    "model__max_iter":        [100, 300, 500],
},

"BayesianGMM": {
    "model__n_components":                    randint(5, 20),   # teto, não valor exato
    "model__covariance_type":                 ["full", "tied", "diag", "spherical"],
    "model__weight_concentration_prior_type": ["dirichlet_process", "dirichlet_distribution"],
    "model__weight_concentration_prior":      loguniform(1e-3, 1e3),  # menor = menos clusters
},

# ============================ ESPECTRAL ===================================
"Spectral": {
    "model__n_clusters":    randint(2, 10),
    "model__affinity":      ["nearest_neighbors", "rbf"],
    "model__n_neighbors":   randint(5, 30),           # só para nearest_neighbors
    "model__gamma":         loguniform(1e-3, 1e1),    # só para rbf
    "model__assign_labels": ["kmeans", "discretize", "cluster_qr"],
},

# ============================ DESLOCAMENTO DE MÉDIA =======================
"MeanShift": {
    "model__bandwidth":   uniform(0.3, 3.0),   # None = estimado automaticamente
    "model__bin_seeding": [True, False],
    "model__min_bin_freq": randint(1, 10),
    "model__cluster_all":  [True, False],
},

# ============================ INCREMENTAL =================================
"Birch": {
    "model__n_clusters":       randint(2, 12),   # None devolve os subclusters brutos
    "model__threshold":        uniform(0.1, 1.5),  # raio máximo do subcluster
    "model__branching_factor": randint(20, 100),
},

# ============================ TROCA DE MENSAGENS ==========================
"AffinityPropagation": {
    "model__damping":    uniform(0.5, 0.49),   # 0.5 a 0.99; abaixo disso não converge
    "model__preference": [None, -10, -50, -100],  # mais negativo = menos clusters
    "model__max_iter":   [200, 500],
},
}

In [ ]:
# ==== 5.3 ROTA A — RandomizedSearchCV com CV real =========================
# Válido só para algoritmos com predict: KMeans, MiniBatchKMeans, BisectingKMeans,
# GaussianMixture, BayesianGMM, Birch, MeanShift.
# Ajusta em 4/5 dos dados e pontua no 1/5 restante -> mede generalização de verdade.

FAMILIA = MELHOR_FAMILIA if MELHOR_FAMILIA in ("KMeans", "GaussianMixture", "Birch") else "KMeans"

search = RandomizedSearchCV(
    estimator=PIPELINES[FAMILIA],
    param_distributions=GRIDS[FAMILIA],
    n_iter=50,
    scoring=SCORING,
    refit=METRICA_ALVO,
    cv=5,                       # KFold comum: sem y, não há o que estratificar
    n_jobs=N_JOBS,
    random_state=RANDOM_STATE,
    verbose=1,
    error_score=np.nan,
)
search.fit(X_train)             # <- sem y

print(f"Melhor {METRICA_ALVO}: {search.best_score_:.4f}")
for k, v in sorted(search.best_params_.items()):
    print(f"  {k}: {v}")

best_model = search.best_estimator_

In [ ]:
# ==== 5.4 ROTA B — RandomizedSearchCV sem divisão =========================
# Um "fold" único que treina e pontua na base inteira. Serve para algoritmos
# sem predict. É uma varredura organizada, NÃO uma validação: sempre cruze
# com estabilidade.
#
# ATENÇÃO: o truque antigo cv=[(slice(None), slice(None))] parou de funcionar
# com DataFrame a partir do scikit-learn 1.7 — o indexador interno passou a
# rejeitar slice para selecionar linhas. Use um vetor de índices explícito:

SEM_CV = [(np.arange(len(X_train)), np.arange(len(X_train)))]

search_b = RandomizedSearchCV(
    estimator=PIPELINES["DBSCAN"],
    param_distributions=GRIDS["DBSCAN"],
    n_iter=40,
    scoring=scorer_silhueta,
    cv=SEM_CV,
    refit=False,
    n_jobs=N_JOBS,
    random_state=RANDOM_STATE,
    error_score=np.nan,
)
search_b.fit(X_train)
res_b = (pd.DataFrame(search_b.cv_results_)
         [["params", "mean_test_score"]]
         .sort_values("mean_test_score", ascending=False))
display(res_b.head(10))

In [ ]:
# ==== 5.5 ROTA C — ParameterSampler manual (a mais informativa) ===========
# Laço explícito: avalia TODAS as métricas de uma vez, filtra partições
# degeneradas e mede estabilidade — coisas que o SearchCV não faz sozinho.

def busca_clustering(pipe_base, grade, X, n_iter=40, seed=RANDOM_STATE,
                     n_min=2, n_max=15, ruido_max=0.40, com_estabilidade=True):
    prep_local = build_preprocessor("scaled").fit(X)
    X_mat = prep_local.transform(X)
    amostras = list(ParameterSampler(grade, n_iter=n_iter, random_state=seed))

    linhas = []
    for i, params in enumerate(amostras):
        try:
            pipe = clone(pipe_base).set_params(**params)
            labels = np.asarray(pipe.fit_predict(X))
            m = metricas_internas(X_mat, labels)

            # Descarta resultados inúteis antes de gastar tempo com estabilidade
            if not (n_min <= m["n_clusters"] <= n_max) or m["ruido"] > ruido_max:
                continue
            if com_estabilidade:
                m["estabilidade"], _ = estabilidade(pipe, X, n_rodadas=3)
            m["params"] = params
            linhas.append(m)
        except Exception:
            continue

    out = pd.DataFrame(linhas)
    if len(out):
        out["score_composto"] = out["silhueta"].rank(pct=True)
        if "estabilidade" in out:
            out["score_composto"] = (out["silhueta"].rank(pct=True) * 0.5
                                     + out["estabilidade"].rank(pct=True) * 0.5)
        out = out.sort_values("score_composto", ascending=False).reset_index(drop=True)
    return out


res_c = busca_clustering(PIPELINES["HDBSCAN"], GRIDS["HDBSCAN"], X_train, n_iter=30)
display(res_c.head(10).round(4))

# melhor_params = res_c.loc[0, "params"]
# best_model = clone(PIPELINES["HDBSCAN"]).set_params(**melhor_params).fit(X_train)

In [ ]:
# ==== 5.6 Confirmação do modelo escolhido =================================
MELHOR_PIPE = best_model            # troque pelo vencedor da rota que você usou
labels_train = np.asarray(MELHOR_PIPE.fit_predict(X_train))

print("Distribuição dos clusters no treino:")
display(pd.Series(labels_train).value_counts().rename("n").to_frame()
        .assign(pct=lambda d: (d["n"] / d["n"].sum() * 100).round(1)))

est_mean, est_std = estabilidade(MELHOR_PIPE, X_train, n_rodadas=20)
print(f"\nEstabilidade (ARI médio em 20 reamostragens): {est_mean:.3f} +/- {est_std:.3f}")
print("  > 0.75  estrutura sólida")
print("  0.5-0.75 estrutura moderada — cuidado ao usar em decisão")
print("  < 0.5   o algoritmo está inventando os grupos")

---
## 6. A Prova Final

Sem gabarito não existe "acerto". A prova final é outra: **os clusters se repetem em
dados novos, se separam visualmente e descrevem perfis que alguém consegue usar?**

Se houver rótulo verdadeiro (produto conhecido, segmento já definido, dataset de
referência), aí sim entram a tabela de contingência e o ARI/NMI — o análogo direto da
matriz de confusão.

In [ ]:
# ==== 6.1 Atribuição no holdout ===========================================
if hasattr(MELHOR_PIPE, "predict"):
    labels_test = np.asarray(MELHOR_PIPE.predict(X_test))
    print("holdout rotulado por predict()")
else:
    # Sem predict: treina um classificador sobre os rótulos do treino.
    # Isso também transforma a clusterização num modelo aplicável em produção.
    from sklearn.ensemble import RandomForestClassifier
    atribuidor = Pipeline([
        ("prep", build_preprocessor("scaled")),
        ("model", RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE)),
    ])
    mask_tr = labels_train != -1
    atribuidor.fit(X_train[mask_tr], labels_train[mask_tr])
    labels_test = atribuidor.predict(X_test)
    print("holdout rotulado por classificador substituto")

prep_full = build_preprocessor("scaled").fit(X_train)
X_test_mat = prep_full.transform(X_test)

print("\nTreino: ", {k: round(v, 4) if isinstance(v, float) else v
                     for k, v in metricas_internas(X_train_mat, labels_train).items()})
print("Holdout:", {k: round(v, 4) if isinstance(v, float) else v
                   for k, v in metricas_internas(X_test_mat, labels_test).items()})
# Silhueta parecida nos dois = a estrutura generaliza. Queda grande = superajuste.

In [ ]:
# ==== 6.2 Diagrama de silhueta ============================================
# Mostra a qualidade de CADA ponto, não só a média. Barras negativas = pontos
# mais próximos de outro cluster do que do próprio.
mask = labels_train != -1
sil_vals = silhouette_samples(np.asarray(X_train_mat)[mask], labels_train[mask])
media = sil_vals.mean()

fig, ax = plt.subplots(figsize=(8, 6))
y_lower = 10
for c in sorted(set(labels_train[mask])):
    vals = np.sort(sil_vals[labels_train[mask] == c])
    y_upper = y_lower + len(vals)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, vals, alpha=.75)
    ax.text(-0.05, y_lower + len(vals) / 2, str(c))
    y_lower = y_upper + 10
ax.axvline(media, c="red", ls="--", label=f"média = {media:.3f}")
ax.set_xlabel("coeficiente de silhueta"); ax.set_ylabel("clusters")
ax.set_title("Diagrama de silhueta"); ax.legend()
plt.tight_layout(); plt.show()

# Um cluster com barra curta e muitos valores negativos deve ser fundido ou
# dividido — sinal de que o k está errado.

In [ ]:
# ==== 6.3 Perfil dos clusters — a etapa que decide se o modelo serve ======
perfil = X_train.copy()
perfil["cluster"] = labels_train

# (a) Médias por cluster, com referência da base inteira
resumo = perfil.groupby("cluster")[NUM_FEATURES].mean()
resumo["n"] = perfil.groupby("cluster").size()
resumo["pct"] = (resumo["n"] / len(perfil) * 100).round(1)
display(resumo.round(2))

# (b) Desvio em relação à média geral: o que torna cada cluster ESPECIAL
media_geral = X_train[NUM_FEATURES].mean()
desvio = ((resumo[NUM_FEATURES] - media_geral) / X_train[NUM_FEATURES].std()).round(2)
display(desvio.style.background_gradient(cmap="RdBu_r", axis=None)
        if hasattr(desvio, "style") else desvio)
# Leitura: +1,5 = o cluster está 1,5 desvio acima da média naquela variável.

# (c) Composição categórica
for c in CAT_FEATURES:
    display(pd.crosstab(perfil["cluster"], perfil[c], normalize="index").round(3))

# (d) Mapa de calor dos perfis
fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow(desvio.values, cmap="RdBu_r", vmin=-2, vmax=2, aspect="auto")
ax.set_xticks(range(len(NUM_FEATURES)), NUM_FEATURES, rotation=30, ha="right")
ax.set_yticks(range(len(desvio)), [f"cluster {i}" for i in desvio.index])
for i in range(desvio.shape[0]):
    for j in range(desvio.shape[1]):
        ax.text(j, i, f"{desvio.values[i, j]:.1f}", ha="center", va="center", fontsize=9)
plt.colorbar(im, label="desvios-padrão vs. média geral")
ax.set_title("Perfil dos clusters"); plt.tight_layout(); plt.show()

# NOMEIE cada cluster com uma frase de negócio. Um cluster sem nome é um cluster
# que ninguém vai usar.

In [ ]:
# ==== 6.4 Projeção 2D =====================================================
X_all_mat = np.asarray(X_train_mat)

pca2 = PCA(n_components=2, random_state=RANDOM_STATE).fit(X_all_mat)
proj_pca = pca2.transform(X_all_mat)

# t-SNE: só para visualizar. Preserva vizinhança local, distorce distância global.
proj_tsne = TSNE(n_components=2, perplexity=30, init="pca",
                 random_state=RANDOM_STATE).fit_transform(X_all_mat)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, proj, tit in [(axes[0], proj_pca,
                       f"PCA ({pca2.explained_variance_ratio_.sum():.1%} da variância)"),
                      (axes[1], proj_tsne, "t-SNE")]:
    for c in sorted(set(labels_train)):
        m = labels_train == c
        ax.scatter(proj[m, 0], proj[m, 1], s=14, alpha=.7,
                   label="ruído" if c == -1 else f"cluster {c}",
                   c="lightgray" if c == -1 else None)
    ax.set_title(tit); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

# Contribuição de cada variável às componentes principais
display(pd.DataFrame(pca2.components_.T,
                     index=X_train_mat.columns, columns=["PC1", "PC2"]).round(3))

In [ ]:
# ==== 6.5 Tabela de contingência e métricas externas ======================
# Só quando existe um rótulo verdadeiro. É o análogo da matriz de confusão —
# mas SEM diagonal significativa: o cluster 0 não tem nenhuma obrigação de
# corresponder ao grupo 0. O que importa é o padrão de concentração.
y_verdade = GRUPO_REAL.loc[X_train.index].values

cont = pd.crosstab(pd.Series(y_verdade, name="grupo real"),
                   pd.Series(labels_train, name="cluster previsto"))
display(cont)
display((cont / cont.sum(axis=1).values.reshape(-1, 1)).round(3))   # normalizado por linha

fig, ax = plt.subplots(figsize=(6, 4.5))
im = ax.imshow(cont.values, cmap="Blues")
ax.set_xticks(range(cont.shape[1]), cont.columns)
ax.set_yticks(range(cont.shape[0]), cont.index)
ax.set_xlabel("cluster previsto"); ax.set_ylabel("grupo real")
for i in range(cont.shape[0]):
    for j in range(cont.shape[1]):
        ax.text(j, i, cont.values[i, j], ha="center", va="center",
                color="white" if cont.values[i, j] > cont.values.max() / 2 else "black")
plt.colorbar(im); ax.set_title("Tabela de contingência")
plt.tight_layout(); plt.show()

h, c_, v = homogeneity_completeness_v_measure(y_verdade, labels_train)
externas = {
    "ARI (Rand ajustado)":  adjusted_rand_score(y_verdade, labels_train),
    "NMI (info. mútua)":    normalized_mutual_info_score(y_verdade, labels_train),
    "AMI (ajustada)":       adjusted_mutual_info_score(y_verdade, labels_train),
    "Homogeneidade":        h,      # cada cluster contém uma só classe
    "Completude":           c_,     # cada classe cai num só cluster
    "V-measure":            v,      # média harmônica das duas
    "Fowlkes-Mallows":      fowlkes_mallows_score(y_verdade, labels_train),
}
display(pd.Series(externas).round(4).to_frame("valor"))
# ARI: 1 = partições idênticas | 0 = concordância ao acaso | negativo = pior que o acaso.
# Todas essas métricas são INVARIANTES à numeração dos clusters — é por isso que
# se usa ARI/NMI aqui, e não acurácia.

In [ ]:
# ==== 6.6 Modelo final e persistência =====================================
# Re-treino na base COMPLETA: o holdout já cumpriu seu papel de confirmar a estrutura.
modelo_producao = clone(MELHOR_PIPE).fit(X)
labels_final = np.asarray(modelo_producao.fit_predict(X)
                          if not hasattr(modelo_producao, "predict")
                          else modelo_producao.predict(X))

# Para rotular clientes NOVOS quando o algoritmo não tem predict, guarde também
# o classificador substituto (ver 6.1) — é ele que vai para produção.

artefato = {
    "modelo": modelo_producao,
    "familia": MELHOR_FAMILIA,
    "n_clusters": len(set(labels_final[labels_final != -1])),
    "features": {"num": NUM_FEATURES, "cat": CAT_FEATURES},
    "perfil_clusters": resumo.to_dict(),
    "estabilidade_ari": est_mean,
    "silhueta": silhouette_score(np.asarray(prep_full.transform(X))[labels_final != -1],
                                 labels_final[labels_final != -1]),
    "nomes_clusters": {           # preencher depois da leitura de 6.3
        0: "descrever perfil",
        1: "descrever perfil",
        2: "descrever perfil",
    },
    "sklearn_version": __import__("sklearn").__version__,
    "treinado_em": pd.Timestamp.now().isoformat(),
}
joblib.dump(artefato, "modelo_clusterizacao.joblib", compress=3)

# Base rotulada para o negócio
X_rotulado = X.copy()
X_rotulado["cluster"] = labels_final
X_rotulado["segmento"] = X_rotulado["cluster"].map(artefato["nomes_clusters"])
X_rotulado.to_parquet("base_segmentada.parquet", index=False)

# --- Carregar e usar ------------------------------------------------------
# art = joblib.load("modelo_clusterizacao.joblib")
# clusters_novos = art["modelo"].predict(X_novo)

---
## Apêndice — Cheatsheet

### Qual família tentar primeiro
| Situação | Comece por |
|---|---|
| Não faz ideia por onde começar | **KMeans** + curva do cotovelo (é o baseline universal) |
| Não sabe quantos grupos existem | **HDBSCAN**, `BayesianGMM`, `MeanShift` |
| Grupos de formato irregular / não convexo | **DBSCAN**, **HDBSCAN**, `SpectralClustering` |
| Tem outliers e quer isolá-los | **HDBSCAN** / `DBSCAN` (rótulo −1 = ruído) |
| Quer probabilidade de pertencer a cada grupo | **GaussianMixture** (`predict_proba`) |
| Precisa de hierarquia / dendrograma | `AgglomerativeClustering`, `BisectingKMeans` |
| Milhões de linhas | `MiniBatchKMeans`, `Birch` |
| Base pequena (< 5.000) e estrutura complexa | `SpectralClustering`, `AffinityPropagation` |
| Muitas dimensões (> 20) | `PCA`/`UMAP` **antes** + `HDBSCAN` ou `KMeans` |
| Só variáveis categóricas | `KModes` / `KPrototypes` (`pip install kmodes`) |
| Dados de texto (TF-IDF) | `TruncatedSVD` + `KMeans`, `metric="cosine"` |

### Sintomas e correções
| Sintoma | Causa provável | Correção |
|---|---|---|
| Um cluster com 95% dos pontos | escala ou outlier dominando | `StandardScaler`/`QuantileTransformer`; trate outliers |
| DBSCAN devolve tudo como ruído (−1) | `eps` pequeno demais | curva k-distância (4.4); aumente `eps` ou reduza `min_samples` |
| DBSCAN devolve um único cluster | `eps` grande demais | reduza `eps` |
| Silhueta alta mas perfis idênticos | agrupou variável redundante | remova correlacionadas; reduza dimensão |
| Clusters mudam a cada execução | estrutura frágil ou `n_init` baixo | suba `n_init`; teste estabilidade; reconsidere o k |
| Silhueta cresce sempre com k | métrica levando ao trivial | use BIC/estabilidade; corte por interpretabilidade |
| Boa silhueta, negócio não usa | clusters sem tradução prática | menos variáveis, mais próximas da decisão |
| Alta dimensão, tudo equidistante | maldição da dimensionalidade | `PCA`/`UMAP` antes de agrupar |

### Erros que invalidam o resultado
1. **Não escalar** antes de KMeans/DBSCAN/hierárquico — a variável de maior amplitude decide tudo sozinha.
2. **Deixar um ID numérico** entre as variáveis.
3. **Ajustar o `StandardScaler` na base inteira** e depois separar holdout — vazamento.
4. **Escolher o k só pela silhueta** — ela premia partições esféricas; cruze com BIC, estabilidade e interpretação.
5. **Usar t-SNE/UMAP como pré-processamento e reportar a silhueta no espaço projetado** — a projeção infla artificialmente a separação.
6. **Tratar o rótulo do cluster como categoria estável ao longo do tempo** — a numeração muda a cada re-treino; ancore pelo perfil, não pelo número.

### Escolha da métrica de validação
```python
# INTERNAS (o caso normal, sem gabarito)
silhouette_score(X, labels)            # [-1,1]  maior é melhor
calinski_harabasz_score(X, labels)     # [0,inf) maior é melhor
davies_bouldin_score(X, labels)        # [0,inf) MENOR é melhor
gmm.bic(X); gmm.aic(X)                 # menor é melhor (só mistura gaussiana)

# EXTERNAS (só com rótulo verdadeiro)
adjusted_rand_score(y, labels)         # o padrão: corrige o acaso
normalized_mutual_info_score(y, labels)
homogeneity_completeness_v_measure(y, labels)

# ESTABILIDADE (a mais confiável na prática — função `estabilidade` em 4.1)
```